# 06b — Router Training (Vision ON/OFF, New Dataset)

Train a **lightweight multimodal router** that selects the best VLM backend
for each (image, prompt) using the new `router_*_final.parquet` datasets.

**What this notebook does:**

1. Load prepared router datasets (train/val/test)  
2. Build `RouterDataset` and PyTorch DataLoaders  
3. Define `MultimodalRouterModel`:
   - Encoder-only transformer
   - Optional frozen CLIP vision encoder (controlled by `config.use_image`)  
4. Train with:
   - Hard labels (`router_best_model_id`)
   - Optional soft labels (`router_soft_p_*`) via KL loss  
   - Full W&B tracking (optional)  
5. Save best checkpoint  
6. Run a quick inference demo on random test samples

In [ ]:
# Optional: install deps if needed (commented out)
# !pip install torch torchvision transformers datasets wandb seaborn matplotlib --upgrade

import os
from pathlib import Path
from dataclasses import dataclass, asdict
from typing import Optional, List, Dict, Tuple

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import confusion_matrix

import wandb

from transformers import AutoTokenizer, CLIPImageProcessor, CLIPModel

In [ ]:
@dataclass
class RouterConfig:
    """Complete configuration for router training & logging."""
    
    # Paths
    data_root: Path = Path.cwd().parent.parent.parent / "dataset" / "final_dataset"
    image_root: Path = Path.cwd().parent.parent.parent / "dataset" / "which_vlm_data" / "images"
    output_dir: Path = Path.cwd() / "router_training_outputs"
    checkpoint_dir: Path = Path.cwd() / "router_checkpoints"
    
    # File names for router datasets
    train_file: str = "router_train_final.parquet"
    val_file: str   = "router_val_final.parquet"
    test_file: str  = "router_test_final.parquet"
    router_subdir: str = "router_final"   # data_root / router_subdir / train_file
    
    # Model architecture
    vision_encoder_name: str = "openai/clip-vit-base-patch32"  # Small CLIP
    text_tokenizer_name: str = "bert-base-uncased"
    d_model: int = 384
    num_layers: int = 4
    num_heads: int = 6
    ffn_dim: int = 1536
    dropout: float = 0.1
    max_text_length: int = 256
    
    # Vision usage
    use_image: bool = True          # <-- Toggle this to False for text-only router
    freeze_vision: bool = True      # Keep CLIP frozen by default
    
    # Training
    batch_size: int = 32
    num_epochs: int = 10
    learning_rate: float = 1e-4
    weight_decay: float = 0.01
    warmup_ratio: float = 0.1
    gradient_clip_norm: float = 1.0
    
    # Loss configuration
    loss_type: str = "combined"  # "ce", "kl", or "combined"
    ce_weight: float = 0.5       # used if loss_type == "combined"
    kl_weight: float = 0.5
    label_smoothing: float = 0.0
    
    # Data loading
    num_workers: int = 4
    pin_memory: bool = True
    
    # Evaluation / logging
    eval_every_n_steps: int = 500   # optional, unused in simple loop
    save_every_n_steps: int = 1000  # optional, unused in simple loop
    log_every_n_steps: int = 50
    
    # W&B
    use_wandb: bool = True
    wandb_project: str = "vlm-router-training"
    wandb_entity: Optional[str] = None   # Set to your W&B user/org or leave None
    wandb_run_name: Optional[str] = "router_v2_with_vision"  # update for different runs
    
    # Device
    device: str = "cuda" if torch.cuda.is_available() else (
        "mps" if torch.backends.mps.is_available() else "cpu"
    )
    
    # Random seed
    seed: int = 42
    
    def __post_init__(self):
        self.output_dir.mkdir(parents=True, exist_ok=True)
        self.checkpoint_dir.mkdir(parents=True, exist_ok=True)

# Instantiate config
config = RouterConfig()

# Set seeds
torch.manual_seed(config.seed)
np.random.seed(config.seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(config.seed)

print("\n=== Configuration ===")
for k, v in asdict(config).items():
    print(f"{k:25s}: {v}")

In [ ]:
print("Loading prepared router datasets...")

router_dir = config.data_root / config.router_subdir

train_df = pd.read_parquet(router_dir / config.train_file)
val_df   = pd.read_parquet(router_dir / config.val_file)
test_df  = pd.read_parquet(router_dir / config.test_file)

print("\nDataset sizes:")
print(f"  Train: {len(train_df):,}")
print(f"  Val:   {len(val_df):,}")
print(f"  Test:  {len(test_df):,}")

print("\nColumns (train) first 25:")
print(train_df.columns[:25].tolist())

# Sanity check: label column exists
assert "router_best_model_id" in train_df.columns, "router_best_model_id missing!"

In [ ]:
# Detect model names from soft label columns or from param list
soft_label_prefix = "router_soft_p_"
soft_label_cols = [c for c in train_df.columns if c.startswith(soft_label_prefix)]

if len(soft_label_cols) > 0:
    model_names = [c[len(soft_label_prefix):] for c in soft_label_cols]
    print("\nDetected model names from soft label columns:")
    for i, m in enumerate(model_names):
        print(f"  {i}: {m}")
else:
    # Fallback: define manually based on your pipeline
    model_names = [
        "deepseek_ocr",
        "qwen2_5_vl_3b",
        "qwen2_5_vl_7b",
        "qwen3_vl_8b_thinking",
        "gemma_3_27b",
    ]
    print("\nUsing hard-coded model names:")
    for i, m in enumerate(model_names):
        print(f"  {i}: {m}")

num_models = len(model_names)

# Sanity check: label range
print("\nLabel stats:")
print(train_df["router_best_model_id"].value_counts().sort_index())
print(f"\nNum models inferred: {num_models}")

In [ ]:
class RouterDataset(Dataset):
    def __init__(
        self,
        df: pd.DataFrame,
        image_root: Path,
        image_processor: CLIPImageProcessor,
        tokenizer,
        config: RouterConfig,
        model_names: List[str],
    ):
        self.df = df.reset_index(drop=True)
        self.image_root = image_root
        self.image_processor = image_processor
        self.tokenizer = tokenizer
        self.config = config
        self.model_names = model_names
        
        # Prepare soft label columns (if any)
        soft_label_prefix = "router_soft_p_"
        self.soft_label_cols = [f"{soft_label_prefix}{m}" for m in model_names if f"{soft_label_prefix}{m}" in df.columns]
        self.has_soft_labels = len(self.soft_label_cols) == len(model_names)
        
        # For quick stats
        print(f"RouterDataset: {len(self.df):,} samples, use_image={config.use_image}, soft_labels={self.has_soft_labels}")
    
    def __len__(self):
        return len(self.df)
    
    def _load_image(self, row) -> torch.Tensor:
        """Load image from disk (via image_path)."""
        img_path = row.get("image_path", None)
        if img_path is None or not isinstance(img_path, str):
            # Return a dummy image if missing
            pixel_values = torch.zeros(3, 224, 224)
            return pixel_values
        
        full_path = self.image_root / img_path
        if not full_path.exists():
            # Fallback: dummy
            pixel_values = torch.zeros(3, 224, 224)
            return pixel_values
        
        # Use CLIPImageProcessor to load & preprocess
        inputs = self.image_processor(images=str(full_path), return_tensors="pt")
        pixel_values = inputs["pixel_values"].squeeze(0)  # [3, H, W]
        return pixel_values
    
    def _build_router_text(self, row) -> str:
        """Combine metadata + prompt into a single text string."""
        prompt = row["prompt_raw"]
        w = row.get("img_width", None)
        h = row.get("img_height", None)
        ar = row.get("img_aspect_ratio", None)
        len_chars = row.get("txt_prompt_length_chars", None)
        len_words = row.get("txt_prompt_length_words", None)
        
        meta_parts = []
        if len_words is not None:
            meta_parts.append(f"PromptLenWords: {int(len_words)}.")
        if len_chars is not None:
            meta_parts.append(f"PromptLenChars: {int(len_chars)}.")
        if w is not None and h is not None:
            meta_parts.append(f"ImageWidth: {int(w)}. ImageHeight: {int(h)}.")
        if ar is not None:
            meta_parts.append(f"ImageAR: {float(ar):.2f}.")
        
        meta_str = " ".join(meta_parts)
        router_text = f"{meta_str} Question: {prompt}"
        return router_text
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        # 1. Image
        if self.config.use_image:
            pixel_values = self._load_image(row)  # [3, H, W]
        else:
            pixel_values = torch.zeros(3, 224, 224)  # dummy to keep shape consistent
        
        # 2. Text
        router_text = self._build_router_text(row)
        encoding = self.tokenizer(
            router_text,
            padding="max_length",
            truncation=True,
            max_length=self.config.max_text_length,
            return_tensors="pt",
        )
        input_ids = encoding["input_ids"].squeeze(0)         # [T]
        attention_mask = encoding["attention_mask"].squeeze(0)  # [T]
        
        # 3. Labels
        label = int(row["router_best_model_id"])
        
        # 4. Soft labels
        if self.has_soft_labels:
            soft = row[self.soft_label_cols].to_numpy(dtype=np.float32)
            soft_labels = torch.from_numpy(soft)
        else:
            soft_labels = torch.zeros(len(self.model_names), dtype=torch.float32)
        
        sample = {
            "pixel_values": pixel_values,
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "label": label,
            "soft_labels": soft_labels,
        }
        return sample

In [ ]:
print("Loading image processor and tokenizer...")
image_processor = CLIPImageProcessor.from_pretrained(config.vision_encoder_name)
tokenizer = AutoTokenizer.from_pretrained(config.text_tokenizer_name)

print("\nCreating RouterDataset objects...")
train_dataset = RouterDataset(train_df, config.image_root, image_processor, tokenizer, config, model_names)
val_dataset   = RouterDataset(val_df,   config.image_root, image_processor, tokenizer, config, model_names)
test_dataset  = RouterDataset(test_df,  config.image_root, image_processor, tokenizer, config, model_names)

train_loader = DataLoader(
    train_dataset,
    batch_size=config.batch_size,
    shuffle=True,
    num_workers=config.num_workers,
    pin_memory=config.pin_memory,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=config.batch_size,
    shuffle=False,
    num_workers=config.num_workers,
    pin_memory=config.pin_memory,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=config.batch_size,
    shuffle=False,
    num_workers=config.num_workers,
    pin_memory=config.pin_memory,
)

print(f"\nTrain batches: {len(train_loader)}")
print(f"Val batches:   {len(val_loader)}")
print(f"Test batches:  {len(test_loader)}")

In [ ]:
class MultimodalRouterModel(nn.Module):
    """Multimodal router: optional Vision + Text → logits over models."""
    
    def __init__(
        self,
        vision_encoder_name: str,
        text_tokenizer_name: str,
        num_models: int,
        d_model: int = 384,
        num_layers: int = 4,
        num_heads: int = 6,
        ffn_dim: int = 1536,
        dropout: float = 0.1,
        freeze_vision: bool = True,
        use_image: bool = True,
    ):
        super().__init__()
        
        self.use_image = use_image
        
        # Vision encoder (CLIP) if using image
        if use_image:
            print(f"Loading vision encoder: {vision_encoder_name}")
            self.vision_encoder = CLIPModel.from_pretrained(vision_encoder_name)
            if freeze_vision:
                for p in self.vision_encoder.parameters():
                    p.requires_grad = False
            vision_dim = self.vision_encoder.config.projection_dim
            self.vision_proj = nn.Linear(vision_dim, d_model)
        else:
            self.vision_encoder = None
            self.vision_proj = None
        
        # Text embedding (simple embedding + custom Transformer)
        print(f"Loading tokenizer: {text_tokenizer_name}")
        tokenizer = AutoTokenizer.from_pretrained(text_tokenizer_name)
        vocab_size = len(tokenizer)
        
        self.text_embedding = nn.Embedding(vocab_size, d_model)
        self.position_embedding = nn.Embedding(512, d_model)
        
        # CLS token
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model))
        
        # Transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=num_heads,
            dim_feedforward=ffn_dim,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        # Classification head
        self.classifier = nn.Linear(d_model, num_models)
        
        print("MultimodalRouterModel initialized.")
    
    def forward(
        self,
        pixel_values: torch.Tensor,   # [B, 3, H, W]
        input_ids: torch.Tensor,      # [B, T]
        attention_mask: torch.Tensor, # [B, T]
    ) -> torch.Tensor:
        batch_size = input_ids.size(0)
        
        # Vision
        if self.use_image and self.vision_encoder is not None:
            with torch.no_grad() if not any(p.requires_grad for p in self.vision_encoder.parameters()) else torch.enable_grad():
                vision_outputs = self.vision_encoder(pixel_values)
                vision_features = vision_outputs.pooler_output  # [B, vision_dim]
            vision_token = self.vision_proj(vision_features).unsqueeze(1)  # [B, 1, d_model]
        else:
            vision_token = None
        
        # Text
        text_emb = self.text_embedding(input_ids)  # [B, T, d_model]
        seq_length = text_emb.size(1)
        positions = torch.arange(seq_length, device=text_emb.device).unsqueeze(0).expand(batch_size, -1)
        text_emb = text_emb + self.position_embedding(positions)
        
        # Compose sequence: [CLS] (+ [VIMG]) + text
        cls_tokens = self.cls_token.expand(batch_size, -1, -1)  # [B, 1, d_model]
        if vision_token is not None:
            sequence = torch.cat([cls_tokens, vision_token, text_emb], dim=1)  # [B, 2+T, d_model]
            # Build attention mask including CLS+VIMG
            extra_ones = torch.ones((batch_size, 2), dtype=attention_mask.dtype, device=attention_mask.device)
            full_mask = torch.cat([extra_ones, attention_mask], dim=1)  # [B, 2+T]
        else:
            sequence = torch.cat([cls_tokens, text_emb], dim=1)  # [B, 1+T, d_model]
            extra_ones = torch.ones((batch_size, 1), dtype=attention_mask.dtype, device=attention_mask.device)
            full_mask = torch.cat([extra_ones, attention_mask], dim=1)  # [B, 1+T]
        
        src_key_padding_mask = (full_mask == 0)  # True = ignore position
        
        hidden = self.transformer(sequence, src_key_padding_mask=src_key_padding_mask)
        cls_hidden = hidden[:, 0, :]
        logits = self.classifier(cls_hidden)
        
        return logits

print("MultimodalRouterModel class defined.")

In [ ]:
model = MultimodalRouterModel(
    vision_encoder_name=config.vision_encoder_name,
    text_tokenizer_name=config.text_tokenizer_name,
    num_models=num_models,
    d_model=config.d_model,
    num_layers=config.num_layers,
    num_heads=config.num_heads,
    ffn_dim=config.ffn_dim,
    dropout=config.dropout,
    freeze_vision=config.freeze_vision,
    use_image=config.use_image,
)

model = model.to(config.device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal parameters:    {total_params:,}")
print(f"Trainable parameters:{trainable_params:,}")
print(f"Frozen parameters:   {total_params - trainable_params:,}")

optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=config.learning_rate,
    weight_decay=config.weight_decay,
)

scheduler = None  # Placeholder for future schedules

In [ ]:
def compute_losses(
    logits: torch.Tensor,
    labels: torch.Tensor,
    soft_labels: torch.Tensor,
    config: RouterConfig,
) -> Dict[str, torch.Tensor]:
    """Compute CE / KL / combined losses."""
    losses = {}
    
    # Cross-entropy with optional label smoothing
    if config.loss_type in ["ce", "combined"]:
        if config.label_smoothing > 0.0:
            num_classes = logits.size(-1)
            with torch.no_grad():
                true_dist = torch.zeros_like(logits)
                true_dist.fill_(config.label_smoothing / (num_classes - 1))
                true_dist.scatter_(1, labels.unsqueeze(1), 1.0 - config.label_smoothing)
            log_probs = F.log_softmax(logits, dim=-1)
            ce_loss = -(true_dist * log_probs).sum(dim=-1).mean()
        else:
            ce_loss = F.cross_entropy(logits, labels)
        losses["ce_loss"] = ce_loss
    
    # KL divergence to soft labels
    if config.loss_type in ["kl", "combined"] and soft_labels is not None and soft_labels.numel() > 0:
        log_probs = F.log_softmax(logits, dim=-1)
        kl_loss = F.kl_div(log_probs, soft_labels, reduction="batchmean")
        losses["kl_loss"] = kl_loss
    
    # Combine
    if config.loss_type == "ce":
        total = losses["ce_loss"]
    elif config.loss_type == "kl":
        total = losses["kl_loss"]
    else:  # combined
        total = config.ce_weight * losses.get("ce_loss", 0.0) +                 config.kl_weight * losses.get("kl_loss", 0.0)
    
    losses["total_loss"] = total
    return losses

In [ ]:
def train_one_epoch(model, loader, optimizer, config: RouterConfig, epoch: int):
    model.train()
    device = config.device
    
    running_loss = 0.0
    running_ce = 0.0
    running_kl = 0.0
    running_correct = 0
    running_total = 0
    
    step = 0
    
    for batch in tqdm(loader, desc=f"Train epoch {epoch}", leave=False):
        pixel_values = batch["pixel_values"].to(device)
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)
        soft_labels = batch["soft_labels"].to(device)
        
        optimizer.zero_grad()
        
        logits = model(pixel_values, input_ids, attention_mask)
        preds = logits.argmax(dim=-1)
        
        # Normalize soft labels if present
        if soft_labels.sum(dim=-1).max() > 1.0 + 1e-3:
            soft_labels = soft_labels / (soft_labels.sum(dim=-1, keepdim=True) + 1e-8)
        
        losses = compute_losses(logits, labels, soft_labels, config)
        loss = losses["total_loss"]
        loss.backward()
        
        if config.gradient_clip_norm is not None and config.gradient_clip_norm > 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), config.gradient_clip_norm)
        
        optimizer.step()
        
        running_loss += loss.item() * labels.size(0)
        running_ce += losses.get("ce_loss", torch.tensor(0.0)).item() * labels.size(0)
        running_kl += losses.get("kl_loss", torch.tensor(0.0)).item() * labels.size(0)
        running_correct += (preds == labels).sum().item()
        running_total += labels.size(0)
        
        step += 1
        if step % config.log_every_n_steps == 0:
            avg_loss = running_loss / running_total
            avg_acc = running_correct / running_total
            print(f"[Train] Step {step:5d} - AvgLoss: {avg_loss:.4f}, AvgAcc: {avg_acc:.4f}")
    
    epoch_loss = running_loss / running_total
    epoch_ce = running_ce / running_total
    epoch_kl = running_kl / running_total
    epoch_acc = running_correct / running_total
    
    metrics = {
        "loss": epoch_loss,
        "ce_loss": epoch_ce,
        "kl_loss": epoch_kl,
        "accuracy": epoch_acc,
    }
    return metrics


def eval_one_epoch(model, loader, config: RouterConfig, split_name: str = "val"):
    model.eval()
    device = config.device
    
    running_loss = 0.0
    running_ce = 0.0
    running_kl = 0.0
    running_correct = 0
    running_total = 0
    
    all_labels = []
    all_preds = []
    
    with torch.no_grad():
        for batch in tqdm(loader, desc=f"Eval ({split_name})", leave=False):
            pixel_values = batch["pixel_values"].to(device)
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)
            soft_labels = batch["soft_labels"].to(device)
            
            logits = model(pixel_values, input_ids, attention_mask)
            preds = logits.argmax(dim=-1)
            
            if soft_labels.sum(dim=-1).max() > 1.0 + 1e-3:
                soft_labels = soft_labels / (soft_labels.sum(dim=-1, keepdim=True) + 1e-8)
            
            losses = compute_losses(logits, labels, soft_labels, config)
            
            running_loss += losses["total_loss"].item() * labels.size(0)
            running_ce += losses.get("ce_loss", torch.tensor(0.0)).item() * labels.size(0)
            running_kl += losses.get("kl_loss", torch.tensor(0.0)).item() * labels.size(0)
            running_correct += (preds == labels).sum().item()
            running_total += labels.size(0)
            
            all_labels.append(labels.cpu().numpy())
            all_preds.append(preds.cpu().numpy())
    
    epoch_loss = running_loss / running_total
    epoch_ce = running_ce / running_total
    epoch_kl = running_kl / running_total
    epoch_acc = running_correct / running_total
    
    all_labels = np.concatenate(all_labels)
    all_preds = np.concatenate(all_preds)
    
    metrics = {
        "loss": epoch_loss,
        "ce_loss": epoch_ce,
        "kl_loss": epoch_kl,
        "accuracy": epoch_acc,
        "labels": all_labels,
        "preds": all_preds,
    }
    return metrics

In [ ]:
if config.use_wandb:
    wandb.init(
        project=config.wandb_project,
        entity=config.wandb_entity,
        name=config.wandb_run_name,
        config=asdict(config),
        tags=["router", "multimodal" if config.use_image else "text_only", "vlm"],
    )
    wandb.watch(model, log="all", log_freq=100)
    print(f"\nW&B run initialized: {wandb.run.name}")
    print(f"W&B URL: {wandb.run.url}")
else:
    print("\nW&B disabled (config.use_wandb = False)")

In [ ]:
best_val_accuracy = 0.0
best_checkpoint_path = config.checkpoint_dir / "router_best.pt"

train_losses = []
val_losses = []
train_accs = []
val_accs = []

print("\n" + "="*60)
print("STARTING TRAINING")
print("="*60)

for epoch in range(1, config.num_epochs + 1):
    print(f"\n{'='*60}")
    print(f"Epoch {epoch}/{config.num_epochs}")
    print(f"{'='*60}")
    
    train_metrics = train_one_epoch(model, train_loader, optimizer, config, epoch)
    val_metrics   = eval_one_epoch(model, val_loader, config, split_name="val")
    
    train_losses.append(train_metrics["loss"])
    val_losses.append(val_metrics["loss"])
    train_accs.append(train_metrics["accuracy"])
    val_accs.append(val_metrics["accuracy"])
    
    print(f"\n[Epoch {epoch}] Train Loss: {train_metrics['loss']:.4f}, "
          f"Train Acc: {train_metrics['accuracy']:.4f}")
    print(f"[Epoch {epoch}] Val   Loss: {val_metrics['loss']:.4f}, "
          f"Val   Acc: {val_metrics['accuracy']:.4f}")
    
    if config.use_wandb:
        log_dict = {
            "train/epoch_loss": train_metrics["loss"],
            "train/epoch_accuracy": train_metrics["accuracy"],
            "val/epoch_loss": val_metrics["loss"],
            "val/epoch_accuracy": val_metrics["accuracy"],
        }
        if "ce_loss" in train_metrics:
            log_dict["train/epoch_ce_loss"] = train_metrics["ce_loss"]
            log_dict["val/ce_loss"] = val_metrics["ce_loss"]
        if "kl_loss" in train_metrics:
            log_dict["train/epoch_kl_loss"] = train_metrics["kl_loss"]
            log_dict["val/kl_loss"] = val_metrics["kl_loss"]
        wandb.log(log_dict, step=epoch * len(train_loader))
    
    # Save best
    if val_metrics["accuracy"] > best_val_accuracy:
        best_val_accuracy = val_metrics["accuracy"]
        print(f"\nNew best val accuracy: {best_val_accuracy:.4f} — saving checkpoint to {best_checkpoint_path}")
        torch.save({
            "model_state_dict": model.state_dict(),
            "config": asdict(config),
            "model_names": model_names,
        }, best_checkpoint_path)

print("\nTraining complete.")
print(f"Best validation accuracy: {best_val_accuracy:.4f}")

# Simple training curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(train_losses, label="Train")
axes[0].plot(val_losses, label="Val")
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()

axes[1].plot(train_accs, label="Train")
axes[1].plot(val_accs, label="Val")
axes[1].set_title("Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
print("\nEvaluating best checkpoint on TEST set...")

# Reload best checkpoint (optional, but safer)
ckpt = torch.load(best_checkpoint_path, map_location=config.device)
model.load_state_dict(ckpt["model_state_dict"])

test_metrics = eval_one_epoch(model, test_loader, config, split_name="test")

print(f"\nTest Loss: {test_metrics['loss']:.4f}")
print(f"Test Acc:  {test_metrics['accuracy']:.4f}")

# Confusion matrix
cm = confusion_matrix(test_metrics["labels"], test_metrics["preds"])
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues", ax=ax,
    xticklabels=model_names, yticklabels=model_names
)
ax.set_xlabel("Predicted model")
ax.set_ylabel("True best model")
ax.set_title("Confusion Matrix (Test)")
plt.tight_layout()
plt.show()

if config.use_wandb:
    wandb.log({"test/accuracy": test_metrics["accuracy"]})

In [ ]:
import random
from IPython.display import display, Image as IPyImage

def router_infer_demo(
    model,
    df: pd.DataFrame,
    num_samples: int = 5,
    show_images: bool = True,
):
    model.eval()
    device = config.device
    
    indices = random.sample(range(len(df)), k=min(num_samples, len(df)))
    subset = df.iloc[indices]
    
    for i, row in subset.iterrows():
        print("\n" + "-"*80)
        print(f"sample_id: {row['sample_id']}")
        print(f"source_dataset: {row.get('source_dataset', 'N/A')}")
        print(f"router_task: {row.get('router_task', 'N/A')}")
        print(f"Ground truth best model: {model_names[row['router_best_model_id']]}")

        if show_images and isinstance(row.get('image_path', None), str):
            img_path = config.image_root / row['image_path']
            if img_path.exists():
                display(IPyImage(filename=str(img_path)))
        
        print(f"\nPrompt:\n{row['prompt_raw']}\n")
        
        # Use dataset to reconstruct sample
        idx_in_dataset = subset.index.get_loc(i)
        sample = test_dataset[idx_in_dataset]
        
        pixel_values = sample["pixel_values"].unsqueeze(0).to(device)
        input_ids = sample["input_ids"].unsqueeze(0).to(device)
        attention_mask = sample["attention_mask"].unsqueeze(0).to(device)
        
        with torch.no_grad():
            logits = model(pixel_values, input_ids, attention_mask)
            probs = F.softmax(logits, dim=-1).squeeze(0).cpu().numpy()
            pred_id = int(probs.argmax())
        
        print(f"Router prediction: {model_names[pred_id]}")
        print("Probabilities:")
        for j, m in enumerate(model_names):
            print(f"  {m:25s}: {probs[j]:.3f}")

# Run demo
router_infer_demo(model, test_df, num_samples=5, show_images=False)

In [ ]:
if config.use_wandb:
    wandb.finish()
    print("\nW&B run finished.")
else:
    print("\nW&B was disabled; nothing to finish.")